In [ ]:
!pip -q install yfinance ta scikit-learn --upgrade

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.4/123.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 77.2 MB/s eta 0:00:00


In [ ]:
import json, os
import pandas as pd
from datetime import datetime
import yfinance as yf
from ta.momentum import RSIIndicator
from ta.trend import EMAIndicator, MACD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

TICKERS = {"^BVSP":"ibov","BOVA11.SA":"bova11"}
px = yf.download(list(TICKERS.keys()),
                 start="2018-01-01",
                 end=datetime.today().strftime("%Y-%m-%d"),
                 auto_adjust=True)["Close"].rename(columns=TICKERS).dropna()
s = px["ibov"].to_frame("close")
s["ret1"] = s["close"].pct_change()
s["ema_10"] = EMAIndicator(s["close"],10).ema_indicator()
s["ema_20"] = EMAIndicator(s["close"],20).ema_indicator()
s["rsi_14"] = RSIIndicator(s["close"],14).rsi()
m = MACD(s["close"],26,12,9); s["macd"]=m.macd(); s["macd_signal"]=m.macd_signal()

for h in [1,5]:
    s[f"fwd_ret_{h}d"] = s["close"].pct_change(h).shift(-h)
    s[f"y_{h}d"] = (s[f"fwd_ret_{h}d"] > 0).astype(int)
s = s.dropna()
features = ["ret1","ema_10","ema_20","rsi_14","macd","macd_signal"]

results = {}
for h in [1,5]:
    df = s.dropna(subset=[f"y_{h}d"])
    split = int(len(df)*0.8)
    tr, te = df.iloc[:split], df.iloc[split:]
    Xtr, ytr = tr[features].values, tr[f"y_{h}d"].values
    Xte, yte = te[features].values, te[f"y_{h}d"].values
    clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
    proba = clf.predict_proba(Xte)[:,1]
    results[f"{h}d"] = {"AUC": float(roc_auc_score(yte, proba)),
                        "MDA": float(accuracy_score(yte, (proba>0.5).astype(int)))}
print(json.dumps(results, indent=2))

[*********************100%***********************]  2 of 2 completed

{
  "1d": {
    "AUC": 0.49824611845888445,
    "MDA": 0.4959785522788204
  },
  "5d": {
    "AUC": 0.5543703277745831,
    "MDA": 0.514745308310992
  }
}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = "/content/drive/MyDrive/TCC_USP"
!ls -la "$DATA_PATH"

Mounted at /content/drive
total 12
drwx------ 2 root root 4096 Sep 19 00:19 data_interim
drwx------ 2 root root 4096 Sep 19 00:19 data_processed
drwx------ 2 root root 4096 Sep 19 00:18 data_raw


In [ ]:
import os

DATA_PATH = "/content/drive/MyDrive/TCC_USP"
RAW_PATH = os.path.join(DATA_PATH, "data_raw")
INTERIM_PATH = os.path.join(DATA_PATH, "data_interim")
PROCESSED_PATH = os.path.join(DATA_PATH, "data_processed")

os.makedirs(RAW_PATH, exist_ok=True)
os.makedirs(INTERIM_PATH, exist_ok=True)
os.makedirs(PROCESSED_PATH, exist_ok=True)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Caminho base do seu projeto
DATA_PATH = "/content/drive/MyDrive/TCC_USP"

Mounted at /content/drive
